# 05 — Baseline × Thompson Sampling

Esta etapa compara uma política determinística (baseline) com um algoritmo adaptativo (**Thompson Sampling**) para decidir o **canal de contato** (`cellular` vs `telephone`).

## Formulação

| Elemento | Definição |
|---|---|
| Braços | `cellular`, `telephone` |
| Recompensa | conversão (`y=1`) |
| Baseline legado | sempre `telephone` (canal fixo operacional) |
| Baseline histórico | sempre o melhor braço empírico do train (`cellular`) |
| Adaptativo | Thompson Sampling com priors Beta(1,1) |

Como o dataset é offline (só observamos o canal já usado), treinamos um **modelo de recompensa** `P(y | contexto, canal)` na Gold train e usamos esse modelo para simular recompensas contrafactuais no conjunto de teste.



In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.bandit import (
    DEFAULT_ARMS,
    build_reward_model,
    predict_arm_probabilities,
    run_fixed_policy,
    run_random_policy,
    run_thompson_sampling,
)

sns.set_theme(style="whitegrid")
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

train = pd.read_csv(ROOT / "data/gold/bank_marketing_gold_train.csv", sep=";")
test = pd.read_csv(ROOT / "data/gold/bank_marketing_gold_test.csv", sep=";")

print(f"Train: {train.shape} | Test: {test.shape}")
print("Conversão empírica por canal (train):")
display(train.groupby("contact")["y"].agg(["mean", "count"]).round(4))


## 1. Modelo de recompensa (simulador contrafactual)



In [ ]:
reward_model = build_reward_model(train)
arm_probs = predict_arm_probabilities(reward_model, test, arms=DEFAULT_ARMS)

summary_p = pd.DataFrame({
    "arm": list(arm_probs.keys()),
    "mean_predicted_p": [float(np.mean(v)) for v in arm_probs.values()],
}).sort_values("mean_predicted_p", ascending=False)

display(summary_p)

best_historical_arm = (
    train.groupby("contact")["y"].mean().sort_values(ascending=False).index[0]
)
print(f"Melhor braço histórico (train): {best_historical_arm}")


## 2. Simulação offline das políticas



In [ ]:
n_rounds = len(test)

baseline_legacy = run_fixed_policy(
    "baseline_legacy_telephone", "telephone", arm_probs, rng, n_rounds
)
baseline_hist = run_fixed_policy(
    "baseline_historical_best", best_historical_arm, arm_probs, np.random.default_rng(RANDOM_SEED + 1), n_rounds
)
baseline_random = run_random_policy(
    DEFAULT_ARMS, arm_probs, np.random.default_rng(RANDOM_SEED + 2), n_rounds
)
ts_result, ts_policy = run_thompson_sampling(
    arm_probs,
    np.random.default_rng(RANDOM_SEED + 3),
    arms=DEFAULT_ARMS,
    n_rounds=n_rounds,
)

results = [baseline_legacy, baseline_random, baseline_hist, ts_result]

metrics = pd.DataFrame([
    {
        "policy": r.policy_name,
        "conversion": r.conversion,
        "exploration_rate": r.exploration_rate,
        **{f"share_{k}": v for k, v in r.arm_share().items()},
    }
    for r in results
]).sort_values("conversion", ascending=False)

display(metrics.round(4))

lift_vs_legacy = ts_result.conversion - baseline_legacy.conversion
lift_vs_random = ts_result.conversion - baseline_random.conversion
print(f"Lift TS vs baseline legado (telephone): {lift_vs_legacy:+.4f}")
print(f"Lift TS vs random: {lift_vs_random:+.4f}")
print(f"TS vs melhor histórico ({best_historical_arm}): {ts_result.conversion - baseline_hist.conversion:+.4f}")
print("\nPosterior Thompson Sampling:")
display(pd.DataFrame(ts_policy.to_dict()))


## 3. Conversão acumulada



In [ ]:
plt.figure(figsize=(10, 5))
for result, color in zip(
    results,
    ["#F58518", "#54A24B", "#4C78A8", "#E45756"],
):
    plt.plot(result.cumulative_conversion, label=result.policy_name, color=color, linewidth=2)

plt.xlabel("Rodada")
plt.ylabel("Conversão acumulada")
plt.title("Baseline × Thompson Sampling — conversão acumulada")
plt.legend()
plt.tight_layout()
plt.show()


## 4. Distribuição de braços escolhidos pelo Thompson Sampling



In [ ]:
share = pd.Series(ts_result.choices).value_counts(normalize=True).rename("share").to_frame()
display(share.round(4))

ax = share["share"].plot(kind="bar", color="#E45756", figsize=(5, 3))
ax.set_title("Participação dos braços — Thompson Sampling")
ax.set_ylabel("share")
ax.set_xlabel("arm")
plt.tight_layout()
plt.show()


## 5. Interpretação

- O **baseline legado** (`telephone`) representa uma regra fixa de canal e obtém a menor conversão.
- O **Thompson Sampling** explora no início e concentra a política em `cellular`, superando o baseline legado e o aleatório.
- A política adaptativa se aproxima do **melhor braço histórico**, sem precisar congelar a regra a priori.

Artefatos salvos em `artifacts/` para o serviço (S3) e tracking MLflow (S3).



In [ ]:
artifacts_dir = ROOT / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(reward_model, artifacts_dir / "reward_model.joblib")

policy_path = artifacts_dir / "thompson_policy.json"
with open(policy_path, "w", encoding="utf-8") as f:
    json.dump(ts_policy.to_dict(), f, indent=4)

metrics_path = artifacts_dir / "bandit_metrics.json"
payload = {
    "random_seed": RANDOM_SEED,
    "n_rounds": n_rounds,
    "arms": list(DEFAULT_ARMS),
    "best_historical_arm": best_historical_arm,
    "metrics": metrics.to_dict(orient="records"),
    "lift_ts_vs_legacy": float(lift_vs_legacy),
    "lift_ts_vs_random": float(lift_vs_random),
    "thompson_policy": ts_policy.to_dict(),
}
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=4)

print(f"Salvo: {artifacts_dir / 'reward_model.joblib'}")
print(f"Salvo: {policy_path}")
print(f"Salvo: {metrics_path}")
